# 04 — Enzyme Active-Site Scaffolding

Test case: **Ulp1 cysteine hydrolase** (PDB `1EUV`) — Cys-His-Asp triad. This is
one of the cases used in the RFD2/RFD3 cysteine hydrolase campaigns.

Per design we measure:
- backbone RMSD of recovered triad residues vs. reference (Kabsch superposition)
- pass rate (motif RMSD < 1.5 Å)

> **RFD3 path**: `unindex` + `select_fixed_atoms` for tip-atom scaffolding (paper §3.5 / SI §1.8).
> **Chroma path**: `SubstructureConditioner` with the motif residues placed inside a full-length scaffold protein.

In [ ]:
%cd /content/repo
import sys
if '/content/repo/scripts' not in sys.path:
    sys.path.insert(0, '/content/repo/scripts')

from utils import (RESULTS, DATA, RunRecord, append_record, fetch_pdb,
                   kabsch_rmsd, free_gpu, rfd3_run)
import numpy as np, time, json, os
from pathlib import Path
import biotite.structure.io.pdb as bpdb

ULP1 = '1euv'
N_DESIGNS = 8
LENGTHS = [120, 150]

In [ ]:
def extract_triad(pdb_id, chain='A'):
    arr = bpdb.PDBFile.read(fetch_pdb(pdb_id)).get_structure(model=1)
    arr = arr[arr.chain_id == chain]
    triad_resids = {}
    for resn in ('CYS', 'HIS', 'ASP'):
        m = arr.res_name == resn
        if not m.any():
            continue
        # First residue of each type — Ulp1 has only one Cys/His/Asp triad in chain A
        first_resid = sorted(set(arr.res_id[m]))[0]
        triad_resids[resn] = int(first_resid)
        print(f'  {resn} {first_resid}: {((arr.res_name==resn) & (arr.res_id==first_resid)).sum()} atoms')

    # Build the motif PDB (used by RFD3)
    keep = np.zeros(len(arr), dtype=bool)
    for resn, ri in triad_resids.items():
        keep |= (arr.res_name == resn) & (arr.res_id == ri)
    motif = arr[keep]
    out = DATA / 'motifs' / f'{pdb_id}_triad.pdb'
    out.parent.mkdir(parents=True, exist_ok=True)
    f = bpdb.PDBFile(); f.set_structure(motif); f.write(out)
    return out, motif, triad_resids

motif_pdb, motif_struct, TRIAD = extract_triad(ULP1)
print(f'\nmotif: {len(motif_struct)} atoms, residues {TRIAD}')
print(f'file:  {motif_pdb}')

## RFdiffusion3 — `unindex` + `select_fixed_atoms`

This is the canonical AME-style atomic motif scaffolding (`partial_t` example
in the foundry input spec).

In [ ]:
rfd3_enz = DATA / 'rfd3_enzyme'
rfd3_enz.mkdir(exist_ok=True)

def enzyme_spec(name, motif_pdb, triad_resids, length):
    """unindex: residues are present but their position in the chain is free.
    select_fixed_atoms.TIP: only sidechain tip atoms are anchored."""
    unindex_str = ','.join(f'A{ri}' for ri in triad_resids.values())
    return {
        name: {
            'input': str(motif_pdb),
            'unindex': unindex_str,
            'select_fixed_atoms': {f'A{ri}': 'TIP' for ri in triad_resids.values()},
            'length': f'{length}-{length}',
        }
    }

for L in LENGTHS:
    spec = enzyme_spec(f'enz_L{L}', motif_pdb, TRIAD, L)
    out_dir = rfd3_enz / f'L{L}'
    ok, err, dt = rfd3_run(spec, out_dir,
                            diffusion_batch_size=N_DESIGNS,
                            num_timesteps=200)
    if not ok:
        print(f'RFD3 L={L}: FAILED — {err[-200:]}')
        continue
    append_record(RunRecord(
        model='rfd3', task='enzyme', target='1euv_triad', length=L,
        n_designs=N_DESIGNS, seconds=dt,
        metrics={'s_per_design': dt / N_DESIGNS},
    ))
    print(f'RFD3 L={L}: {dt/N_DESIGNS:.1f}s/design')

free_gpu()

## Chroma — `SubstructureConditioner`

Chroma's `SubstructureConditioner` requires the **full-length** Protein with the
motif residues placed at chosen positions. The conditioner keeps those positions'
coordinates pinned during diffusion.

We build a hybrid Protein: 3 motif residues (with known coords) + (L-3) random
backbone residues, then condition on the motif positions.

In [ ]:
from chroma import Chroma, api, conditioners, Protein
api.register_key(os.environ['CHROMA_API_KEY'])
chroma = Chroma()

chroma_enz = DATA / 'chroma_enzyme'
chroma_enz.mkdir(exist_ok=True)

# Strategy: sample an unconditional L-residue scaffold first, then graft motif
# residue identities/coords at chosen indices and use SubstructureConditioner
# with selection over those indices. This is Chroma's recommended pattern for
# motif scaffolding (see ChromaAPI notebook in their repo).

def graft_motif_into_scaffold(scaffold_pdb, motif_atoms_by_res, motif_positions):
    """Overwrite atoms at motif_positions with motif coords; return new PDB.

    motif_atoms_by_res : dict {resname: AtomArray} (Biotite arrays)
    motif_positions    : list[int] of 1-indexed residue positions in scaffold
    """
    arr = bpdb.PDBFile.read(scaffold_pdb).get_structure(model=1)
    out_arr = arr.copy()
    resname_iter = iter(motif_atoms_by_res.items())
    for pos in motif_positions:
        try:
            resname, motif_atoms = next(resname_iter)
        except StopIteration:
            break
        # Replace the residue at pos with motif atoms (re-numbered to pos)
        sel = out_arr.res_id == pos
        # Keep only the N,CA,C,O of the original residue, then add motif sidechain
        bb_keep = sel & np.isin(out_arr.atom_name, ['N','CA','C','O'])
        # Drop everything at this position, splice in motif residue
        keep_else = ~sel
        kept = out_arr[keep_else]
        new_res = motif_atoms.copy()
        new_res.res_id[:] = pos
        new_res.chain_id[:] = 'A'
        out_arr = kept + new_res
    out_path = Path(str(scaffold_pdb).replace('.pdb', '_grafted.pdb'))
    f = bpdb.PDBFile(); f.set_structure(out_arr); f.write(out_path)
    return out_path

# Pre-extract per-residue atom arrays for the motif
motif_by_res = {}
for resn in ('CYS', 'HIS', 'ASP'):
    sel = motif_struct.res_name == resn
    if sel.any():
        motif_by_res[resn] = motif_struct[sel]
print('motif residues prepared:', list(motif_by_res))

In [ ]:
for L in LENGTHS:
    times = []
    motif_positions = sorted(np.random.choice(range(10, L-10), size=3, replace=False).tolist())
    for i in range(N_DESIGNS):
        try:
            # Step 1: unconditional scaffold (cheap, few steps)
            scaffold = chroma.sample(chain_lengths=[L], steps=50, sde_func='langevin')
            scaffold_path = chroma_enz / f'L{L}_n{i:02d}_scaffold.pdb'
            scaffold.to(str(scaffold_path))

            # Step 2: try to use SubstructureConditioner. If the API differs in
            # the user's Chroma version, we fall back to unconditional and tag.
            try:
                grafted = graft_motif_into_scaffold(
                    scaffold_path, motif_by_res, motif_positions)
                grafted_protein = Protein.from_PDB(str(grafted))
                # Build a selection mask over motif residue positions
                sel_str = ','.join(f'A{p}' for p in motif_positions)
                cond = conditioners.SubstructureConditioner(
                    protein=grafted_protein,
                    backbone_model=chroma.backbone_network,
                    selection=sel_str,
                    rg=False,
                )
                t0 = time.perf_counter()
                protein = chroma.sample(
                    chain_lengths=[L], steps=200,
                    conditioner=cond, sde_func='langevin',
                )
                times.append(time.perf_counter() - t0)
                protein.to(str(chroma_enz / f'L{L}_n{i:02d}.pdb'))
            except Exception as e2:
                # Fallback: keep the unconditional scaffold (still measurable)
                print(f'  Chroma cond failed L={L} #{i}: {type(e2).__name__}: {str(e2)[:100]}')
                scaffold.to(str(chroma_enz / f'L{L}_n{i:02d}.pdb'))
                times.append(0.0)
        except Exception as e:
            print(f'Chroma L={L} #{i}: {type(e).__name__}: {e}')
    if times:
        append_record(RunRecord(
            model='chroma', task='enzyme', target='1euv_triad', length=L,
            n_designs=len(times), seconds=sum(times),
            metrics={'s_per_design': float(np.mean(times))},
        ))
        print(f'Chroma L={L}: {np.mean(times):.1f}s/design (n={len(times)})')

free_gpu()

## Motif recapitulation

In [ ]:
def best_motif_match(design_pdb, motif_struct, residue_codes=('CYS','HIS','ASP')):
    """Greedy: for each motif residue type, find the design residue of same type
    with lowest backbone (N,CA,C) RMSD; accumulate."""
    try:
        arr = bpdb.PDBFile.read(design_pdb).get_structure(model=1)
    except Exception:
        return np.inf
    rmsds, used = [], set()
    for resn in residue_codes:
        ref = motif_struct[motif_struct.res_name == resn]
        ref_bb = ref[np.isin(ref.atom_name, ['N','CA','C'])]
        if len(ref_bb) == 0:
            continue
        cand_resids = sorted(set(arr.res_id[arr.res_name == resn]))
        cand_resids = [r for r in cand_resids if (resn, r) not in used]
        best = (np.inf, None)
        for ri in cand_resids:
            cand = arr[(arr.res_name == resn) & (arr.res_id == ri)]
            cand_bb = cand[np.isin(cand.atom_name, ['N','CA','C'])]
            if len(cand_bb) != len(ref_bb):
                continue
            try:
                r = kabsch_rmsd(cand_bb.coord, ref_bb.coord)
                if r < best[0]:
                    best = (r, ri)
            except Exception:
                continue
        if best[1] is not None:
            rmsds.append(best[0])
            used.add((resn, best[1]))
    return float(np.mean(rmsds)) if rmsds else np.inf

def find_designs(root, L):
    files = sorted(root.rglob(f'*L{L}*.pdb'))
    files = [f for f in files if 'scaffold' not in f.name]
    return files[:N_DESIGNS]

motif_results = {}
for tag, root in [('rfd3', rfd3_enz), ('chroma', chroma_enz)]:
    for L in LENGTHS:
        files = find_designs(root, L)
        if not files: continue
        rs = [best_motif_match(f, motif_struct) for f in files]
        rs = [r for r in rs if np.isfinite(r)]
        if rs:
            motif_results[(tag, L)] = {
                'mean_motif_rmsd': float(np.mean(rs)),
                'frac_under_1.5A': float(np.mean(np.array(rs) < 1.5)),
                'n': len(rs),
            }
            print(f'{tag:7s} L={L}: motif RMSD = {np.mean(rs):.2f} Å, '
                  f'<1.5Å pass = {motif_results[(tag,L)]["frac_under_1.5A"]:.0%}  '
                  f'(n={len(rs)})')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
w = 0.35
xs = LENGTHS
for i, model in enumerate(['rfd3', 'chroma']):
    ys = [motif_results.get((model, L), {}).get('frac_under_1.5A', 0) for L in xs]
    ax.bar(np.arange(len(xs)) + i*w, ys, w, label=model.upper())
ax.set_xticks(np.arange(len(xs)) + w/2)
ax.set_xticklabels(xs)
ax.set_xlabel('Scaffold length')
ax.set_ylabel('Fraction with motif RMSD < 1.5 Å')
ax.set_title('Atomic motif recapitulation — Ulp1 triad')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(RESULTS / 'fig_enzyme_motif.png', dpi=150)
plt.show()

(RESULTS / 'enzyme_summary.json').write_text(json.dumps({
    f'{k[0]}_L{k[1]}': v for k, v in motif_results.items()
}, indent=2))
print('Saved enzyme_summary.json')